In [0]:
# ============================================================
# Notebook: 03_gold_sales_analytics
# Purpose: Create Gold business aggregates for reporting
# Author: Dele Fatoba
# ============================================================

from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    countDistinct,
    year,
    month,
    round as spark_round
)

silver_table_name = (
    "retaildataops_dbw_dev_v4ptce_7405619702729069."
    "silver."
    "online_retail_transactions"
)


gold_sales_by_country_table = (
    "retaildataops_dbw_dev_v4ptce_7405619702729069."
    "gold."
    "sales_by_country"
)

gold_sales_by_product_table = (
    "retaildataops_dbw_dev_v4ptce_7405619702729069."
    "gold."
    "sales_by_product"
)

gold_monthly_sales_table = (
    "retaildataops_dbw_dev_v4ptce_7405619702729069."
    "gold."
    "monthly_sales"
)

gold_top_customers_table = (
    "retaildataops_dbw_dev_v4ptce_7405619702729069."
    "gold."
    "top_customers"
)


sales_df = spark.table(silver_table_name)

display(sales_df.limit(20))
sales_df.printSchema()

sales_only_df = sales_df.filter(col("transaction_type") == "SALE")

sales_by_country = (
    sales_only_df
    .groupBy("country")
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("total_sales"),
        countDistinct("invoice_no").alias("transaction_count"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(col("total_sales").desc())
)

sales_by_product = (
    sales_only_df
    .groupBy("stock_code", "product_description")
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("total_sales"),
        spark_sum("quantity").alias("total_quantity_sold"),
        countDistinct("invoice_no").alias("transaction_count")
    )
    .orderBy(col("total_sales").desc())
)

monthly_sales = (
    sales_only_df
    .withColumn("sales_year", year(col("invoice_date")))
    .withColumn("sales_month", month(col("invoice_date")))
    .groupBy("sales_year", "sales_month")
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("monthly_sales"),
        countDistinct("invoice_no").alias("monthly_transactions"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy("sales_year", "sales_month")
)

top_customers = (
    sales_only_df
    .filter(col("customer_id").isNotNull())
    .groupBy("customer_id", "country")
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("customer_total_sales"),
        countDistinct("invoice_no").alias("order_count")
    )
    .orderBy(col("customer_total_sales").desc())
)

spark.sql("CREATE SCHEMA IF NOT EXISTS retaildataops_dbw_dev_v4ptce_7405619702729069.gold")

sales_by_country.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_sales_by_country_table)
sales_by_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_sales_by_product_table)
monthly_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_monthly_sales_table)
top_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_top_customers_table)

print("Gold Delta tables created successfully.")

display(spark.table(gold_sales_by_country_table).limit(20))
display(spark.table(gold_sales_by_product_table).limit(20))
display(spark.table(gold_monthly_sales_table).limit(20))
display(spark.table(gold_top_customers_table).limit(20))

invoice_no,stock_code,product_description,quantity,invoice_date,unit_price,customer_id,country,ingestion_timestamp,source_file,transaction_type,sales_amount,silver_processed_timestamp
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,83.4,2026-06-02T14:28:06.788Z
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,81.0,2026-06-02T14:28:06.788Z
489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01T09:06:00.000Z,1.69,13078,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,27.04,2026-06-02T14:28:06.788Z
489437,21351,CINAMMON & ORANGE WREATH,2,2009-12-01T09:08:00.000Z,6.75,15362,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,13.5,2026-06-02T14:28:06.788Z
489437,21987,PACK OF 6 SKULL PAPER CUPS,12,2009-12-01T09:08:00.000Z,0.65,15362,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,7.800000000000001,2026-06-02T14:28:06.788Z
489438,84031A,CHARLIE+LOLA RED HOT WATER BOTTLE,56,2009-12-01T09:24:00.000Z,3.0,18102,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,168.0,2026-06-02T14:28:06.788Z
489439,22333,RETRO SPORT PARTY BAG + STICKER SET,8,2009-12-01T09:28:00.000Z,1.65,12682,FRANCE,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,13.2,2026-06-02T14:28:06.788Z
489441,84029E,RED WOOLLY HOTTIE WHITE HEART.,36,2009-12-01T09:44:00.000Z,2.95,18087,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,106.2,2026-06-02T14:28:06.788Z
489442,22111,SCOTTIE DOG HOT WATER BOTTLE,3,2009-12-01T09:46:00.000Z,4.95,13635,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,14.850000000000001,2026-06-02T14:28:06.788Z
489442,84899E,YELLOW + BROWN BEAR FELT PURSE KIT,12,2009-12-01T09:46:00.000Z,1.25,13635,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,15.0,2026-06-02T14:28:06.788Z


root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- silver_processed_timestamp: timestamp (nullable = true)

Gold Delta tables created successfully.


country,total_sales,transaction_count,unique_customers
UNITED KINGDOM,1.731957183E7,36536,5350
EIRE,657370.09,626,5
NETHERLANDS,553805.51,228,22
GERMANY,423823.42,789,107
FRANCE,350172.97,622,95
AUSTRALIA,169250.26,95,15
SPAIN,108122.23,154,41
SWITZERLAND,100592.24,93,22
SWEDEN,91869.82,105,19
DENMARK,68539.44,43,12


stock_code,product_description,total_sales,total_quantity_sold,transaction_count
M,Manual,330442.96,9195,785
22423,REGENCY CAKESTAND 3 TIER,329456.72,26405,3918
DOT,DOTCOM POSTAGE,309854.11,1415,1415
85123A,WHITE HANGING HEART T-LIGHT HOLDER,255631.68,93467,5356
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.6,80995,1
47566,PARTY BUNTING,147515.98,28046,2674
85099B,JUMBO BAG RED RETROSPOT,145500.94,77060,3245
84879,ASSORTED COLOUR BIRD ORNAMENT,128787.07,79764,2807
POST,POSTAGE,125682.42,5363,1851
22086,PAPER CHAIN KIT 50'S CHRISTMAS,116999.54,34839,2018


sales_year,sales_month,monthly_sales,monthly_transactions,unique_customers
2009,12,817595.24,1682,955
2010,1,648755.69,1105,720
2010,2,549730.72,1202,772
2010,3,828231.82,1681,1057
2010,4,676363.13,1462,942
2010,5,652946.23,1500,966
2010,6,744920.22,1645,1041
2010,7,646409.43,1529,928
2010,8,692259.79,1425,911
2010,9,917373.47,1839,1145


customer_id,country,customer_total_sales,order_count
18102,UNITED KINGDOM,580987.04,145
14646,NETHERLANDS,528379.98,151
14156,EIRE,313411.62,156
14911,EIRE,291420.81,398
17450,UNITED KINGDOM,244784.25,51
13694,UNITED KINGDOM,195640.69,143
17511,UNITED KINGDOM,172132.87,60
16446,UNITED KINGDOM,168472.5,2
16684,UNITED KINGDOM,147142.77,55
12415,AUSTRALIA,144458.37,28
